# 04 · Detekcja mowy nienawiści – HerBERT-large (Polish Hate Speech)

Model: [`dkleczek/Polish-Hate-Speech-Detection-Herbert-Large`](https://huggingface.co/dkleczek/Polish-Hate-Speech-Detection-Herbert-Large) – HerBERT-large douczony na danych PolEval 2019 (mowa nienawiści).

Ten sam przebieg co w poprzednim projekcie (AMC_Hate_Speech, notebooki 04–07), na komentarzach, których używa zespół:

1. **Zbiór komentarzy** = filtr tematyczny zespołu (`scripts/topic_filter.py`): komentarz zostaje, gdy słownik obelg daje 1/1
   **albo** `scripts/topic_sample.is_potential_migration` znajduje temat migracji we wpisie lub komentarzu. Liczone w pamięci
   tym samym kodem – nic nie jest zapisywane do `topic/`.
2. **Najpierw słownik** (`src/lexicon/slurs.py`, ta sama reguła co `scripts/preprocess_slurs.py`, codebook 4.12):
   `hate_auto = 1` → hejt, model nie jest pytany.
3. **Model tylko dla pozostałych komentarzy.** Teksty dłuższe niż `MAX_LENGTH` tokenów są dzielone na nakładające się okna,
   `prob_hate_max` = maksimum po oknach.
4. `hate_final` = 1, gdy słownik **albo** model (`prob_hate_max ≥ THRESHOLD`); `label_source` = `slownik` / `{model}`.

**Wyjście:** `data/interim/<platforma>/<wydarzenie>/{OUT_DIRNAME}/comments.csv` (+ `meta.json`) oraz
`data/interim/<platforma>/{OUT_DIRNAME}_summary.csv`. Kolumny: `comment_id, in_topic, hate_auto, hate_migrants_auto, slurs,
slur_categories, slur_needs_context, prob_hate, prob_hate_max, hate_pred_max, n_tokens, n_windows, hate_final, label_source`.
Pliki nie zawierają treści komentarzy (tylko id i wyniki).

**Przerwanie:** wyniki zapisują się co `CHUNK_SIZE` tekstów (`_parts/`); po przerwaniu odpal notebook jeszcze raz.
Gotowe wydarzenia (zgodny `meta.json`) są pomijane, `FORCE = True` liczy od nowa.

In [1]:
# ── KONFIGURACJA ──────────────────────────────────────────────────────────────
PLATFORMS = ("wykop", "bluesky", "youtube")
EVENTS = None                 # None = wszystkie wydarzenia z clean/; albo np. ["2024-06-06_Smierc_Zolnierza"]
MODEL_ID = "dkleczek/Polish-Hate-Speech-Detection-Herbert-Large"
OUT_DIRNAME = "herbert"
HATE_LABEL_INDEX = 1          # model nie ma id2label; 1 = hate (PolEval 2019)
THRESHOLD = 0.5               # prob_hate_max >= THRESHOLD -> hejt

MAX_LENGTH = 512              # tokeny na okno
TOKENS_PER_BATCH = 8192       # budżet tokenów na batch (mniej = mniej pamięci)
MAX_BATCH_SIZE = 128
WINDOW_OVERLAP = 128          # nakładka okien dla długich tekstów
CHUNK_SIZE = 2000             # co tyle tekstów zapis na dysk
USE_FP16 = True               # half precision na GPU/MPS (wyłącza się sam, gdy zmienia wyniki)
FORCE = False                 # True = policz od nowa

In [8]:
# Jednorazowo, jeśli brakuje pakietów:
%pip install "torch>=2.6" transformers tqdm sacremoses

  Using cached sacremoses-0.2.0-py3-none-any.whl (867 kB)
  Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
You should consider upgrading via the '/Users/gosia/Desktop/SZT_SEM_2/AMC/AMC_FINAL/Social_Media_Hate_Speech/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import json, os, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "events.json").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.classifier.hate_models import Predictor, count_todo, event_dirs, run_event

def _read_env(path):
    env = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                env[k.strip()] = v.strip().strip('"').strip("'")
    return env

_env = _read_env(ROOT / ".env")
HF_TOKEN = _env.get("TOKEN") or _env.get("HF_TOKEN") or os.environ.get("HF_TOKEN")

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
    torch.set_num_threads(os.cpu_count() or 4)

DIRS = event_dirs(PLATFORMS, EVENTS)
print(f"projekt : {ROOT}")
print(f"device  : {DEVICE} | torch {torch.__version__}")
print(f"wydarzenia: {len(DIRS)} ({', '.join(f'{p}: {sum(1 for k in DIRS if k[0] == p)}' for p in PLATFORMS)})")

PREP = None                   # tekst z clean/ bez zmian (jak w notebooku 04)
TOKEN = HF_TOKEN
SIGNATURE = {"model": MODEL_ID, "max_length": MAX_LENGTH, "hate_label_index": HATE_LABEL_INDEX, "topic": "topic_filter.py", "pipeline": "slownik->model v1"}

projekt : /Users/gosia/Desktop/SZT_SEM_2/AMC/AMC_FINAL/Social_Media_Hate_Speech
device  : mps | torch 2.8.0
wydarzenia: 37 (wykop: 11, bluesky: 11, youtube: 15)


/Users/gosia/Desktop/SZT_SEM_2/AMC/AMC_FINAL/Social_Media_Hate_Speech/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/gosia/Desktop/SZT_SEM_2/AMC/AMC_FINAL/Social_Media_Hate_Speech/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 1. Zbiór komentarzy i ile pracy dla modelu

In [4]:
# zbiór komentarzy (filtr tematyczny zespołu), ile rozstrzyga słownik, ile trafia do modelu
status = count_todo(DIRS, OUT_DIRNAME, SIGNATURE, force=FORCE)
display(status)
display(status.groupby("platform")[["clean", "topic", "slownik_1_1", "do_modelu"]].sum())
print(f"Do policzenia modelem: {status['do_modelu'].sum():,} komentarzy")

,platform,event,clean,topic,slownik_1_1,do_modelu
0,wykop,2024-04-10_Pakt_Azyl_PE,24718,17917,499,17418
1,wykop,2024-05-28_Atak_Zolnierz_Granica,25349,20946,566,20380
2,wykop,2024-06-06_Smierc_Zolnierza,24841,20542,531,20011
3,wykop,2024-09-08_Bojka_Srem,19293,14799,470,14329
4,wykop,2024-10-12_Strategia_Migracyjna,23861,18515,606,17909
5,wykop,2025-06-06_Naloty_ICE_Los_Angeles,24340,18491,451,18040
6,wykop,2025-10-15_Dezinformacja_Krakow,18301,13571,300,13271
7,wykop,2026-06-12_Pakt_Migracyjny_Wejscie_w_Zycie,24516,18843,793,18050
8,wykop,2026-07-30_Ceuta_Masowe_Przekroczenie,24634,19312,689,18623
9,wykop,2026-08-15_Atak_Kamienna_Gora,24048,18539,534,18005


,clean,topic,slownik_1_1,do_modelu
platform,,,,
bluesky,175103,46959,239,46720
wykop,255857,198151,5826,192325
youtube,1228811,207345,11205,196140


Do policzenia modelem: 435,185 komentarzy


## 2. Model, sanity check i szacowany czas

In [9]:
SANITY = [
    "Dzień dobry, jutro ma padać deszcz.",
    "Bardzo ciekawy artykuł, dzięki za wrzutkę.",
    "Uchodźcy powinni dostać pomoc, to ludzie jak my.",
    "Wszystkich tych brudasów trzeba wywalić z kraju, to zwierzęta.",
    "Ty pierdolony debilu, zdychaj.",
    "Imigranci to robactwo, które trzeba wytępić.",
]
t0 = time.time()
predictor = Predictor(MODEL_ID, DEVICE, prep=PREP, max_length=MAX_LENGTH, tokens_per_batch=TOKENS_PER_BATCH,
                      max_batch_size=MAX_BATCH_SIZE, window_overlap=WINDOW_OVERLAP, hate_label_index=HATE_LABEL_INDEX,
                      token=TOKEN, use_fp16=USE_FP16)
p, *_ = predictor.predict(SANITY)
print(f"model załadowany w {time.time() - t0:.1f}s | precyzja: {predictor.precision}")
display(pd.DataFrame({"tekst": SANITY, "prob_hate": p.round(3)}))
print("Ostatnie 3 powinny mieć WYSOKIE prob_hate, pierwsze 3 NISKIE – inaczej zmień HATE_LABEL_INDEX.")

# benchmark -> ETA
sample = []
for d in list(DIRS.values())[:8]:
    t = pd.read_csv(d / "clean" / "comments.csv", encoding="utf-8-sig", usecols=["text"], dtype=str,
                    keep_default_na=False)["text"]
    sample += t[t.str.strip() != ""].sample(min(64, len(t)), random_state=0).tolist()
predictor.predict(sample[:16])
t0 = time.time(); predictor.predict(sample)
rate = len(sample) / (time.time() - t0)
print(f"~{rate:.0f} tekstów/s -> szacunkowo {status['do_modelu'].sum() / rate / 60:.0f} min dla wszystkich platform")

model załadowany w 2.6s | precyzja: float16


,tekst,prob_hate
0,"Dzień dobry, jutro ma padać deszcz.",0.049
1,"Bardzo ciekawy artykuł, dzięki za wrzutkę.",0.071
2,"Uchodźcy powinni dostać pomoc, to ludzie jak my.",0.117
3,Wszystkich tych brudasów trzeba wywalić z kraj...,0.948
4,"Ty pierdolony debilu, zdychaj.",0.918
5,"Imigranci to robactwo, które trzeba wytępić.",0.867


Ostatnie 3 powinny mieć WYSOKIE prob_hate, pierwsze 3 NISKIE – inaczej zmień HATE_LABEL_INDEX.
~490 tekstów/s -> szacunkowo 15 min dla wszystkich platform


## 3. Predykcje (można przerwać i wznowić)

In [10]:
total = tqdm(total=int(status["do_modelu"].sum()), desc="WSZYSTKO", unit="txt", smoothing=0.05)
results = []
try:
    for (platform, ev), d in DIRS.items():
        r = run_event(predictor, d, OUT_DIRNAME, SIGNATURE, threshold=THRESHOLD, chunk_size=CHUNK_SIZE,
                      force=FORCE, total_bar=total, tqdm=tqdm)
        results.append(r)
        total.write(f"✔ {platform}/{ev}: {r['status']} | hate_final {r['hate_final']:,}/{r['topic']:,}")
except KeyboardInterrupt:
    total.write(f"⏸ Przerwano – policzone fragmenty są w */{OUT_DIRNAME}/_parts, odpal komórkę ponownie.")
finally:
    total.close()

WSZYSTKO:   4%|▍         | 17418/435185 [00:25<10:14, 680.16txt/s]

✔ wykop/2024-04-10_Pakt_Azyl_PE: policzone | hate_final 5,568/17,917


WSZYSTKO:   9%|▊         | 37798/435185 [00:57<10:08, 652.79txt/s]

✔ wykop/2024-05-28_Atak_Zolnierz_Granica: policzone | hate_final 7,149/20,946


WSZYSTKO:  13%|█▎        | 57809/435185 [01:29<09:49, 640.03txt/s]

...

✔ youtube/2026-07-30_Ceuta_Masowe_Przekroczenie: policzone | hate_final 7,312/16,743


WSZYSTKO:  95%|█████████▌| 413782/435185 [21:19<00:53, 403.50txt/s]

✔ youtube/2026-08-15_Atak_Kamienna_Gora: policzone | hate_final 6,441/14,024


WSZYSTKO: 100%|██████████| 435185/435185 [22:05<00:00, 328.42txt/s]

✔ youtube/2026-09-15_Atak_Jaroslaw: policzone | hate_final 9,413/22,878


## 4. Podsumowanie

In [11]:
summary = pd.DataFrame(results)
if len(summary):
    cols = ["platform", "event", "clean", "topic", "slownik_1_1", "model_scored", "model_hate", "hate_final",
            "share_hate_final", "windowed", "lexicon_from"]
    summary = summary[cols]
    for platform, s in summary.groupby("platform"):
        p = ROOT / "data" / "interim" / platform / f"{OUT_DIRNAME}_summary.csv"
        s.to_csv(p, index=False, encoding="utf-8-sig")
        print("zapisano:", p.relative_to(ROOT))
    display(summary)
    display(summary.groupby("platform")[["topic", "slownik_1_1", "model_hate", "hate_final"]].sum()
            .assign(udzial_hate=lambda d: (d["hate_final"] / d["topic"]).round(4)))

zapisano: data/interim/bluesky/herbert_summary.csv
zapisano: data/interim/wykop/herbert_summary.csv
zapisano: data/interim/youtube/herbert_summary.csv


,platform,event,clean,topic,slownik_1_1,model_scored,model_hate,hate_final,share_hate_final,windowed,lexicon_from
0,wykop,2024-04-10_Pakt_Azyl_PE,24718,17917,499,17418,5069,5568,0.3108,22,slurs/ (DVC)
1,wykop,2024-05-28_Atak_Zolnierz_Granica,25349,20946,566,20380,6583,7149,0.3413,18,slurs/ (DVC)
2,wykop,2024-06-06_Smierc_Zolnierza,24841,20542,531,20011,6562,7093,0.3453,19,slurs/ (DVC)
3,wykop,2024-09-08_Bojka_Srem,19293,14799,470,14329,4384,4854,0.3280,19,slurs/ (DVC)
4,wykop,2024-10-12_Strategia_Migracyjna,23861,18515,606,17909,5746,6352,0.3431,24,slurs/ (DVC)
5,wykop,2025-06-06_Naloty_ICE_Los_Angeles,24340,18491,451,18040,5330,5781,0.3126,30,slurs/ (DVC)
6,wykop,2025-10-15_Dezinformacja_Krakow,18301,13571,300,13271,3934,4234,0.3120,21,slurs/ (DVC)
7,wykop,2026-06-12_Pakt_Migracyjny_Wejscie_w_Zycie,24516,18843,793,18050,5964,6757,0.3586,22,slurs/ (DVC)
8,wykop,2026-07-30_Ceuta_Masowe_Przekroczenie,24634,19312,689,18623,6133,6822,0.3533,22,slurs/ (DVC)
9,wykop,2026-08-15_Atak_Kamienna_Gora,24048,18539,534,18005,5868,6402,0.3453,21,slurs/ (DVC)


,topic,slownik_1_1,model_hate,hate_final,udzial_hate
platform,,,,,
bluesky,46959,239,8856,9095,0.1937
wykop,198151,5826,60888,66714,0.3367
youtube,207345,11205,80684,91889,0.4432


In [12]:
def load_with_text(platform, event_folder):
    d = ROOT / "data" / "interim" / platform / event_folder
    clean = pd.read_csv(d / "clean" / "comments.csv", encoding="utf-8-sig", dtype={"comment_id": str},
                        usecols=["comment_id", "relative_day", "text"])
    pred = pd.read_csv(d / OUT_DIRNAME / "comments.csv", encoding="utf-8-sig", dtype={"comment_id": str})
    return pred.merge(clean, on="comment_id", how="left")

pd.set_option("display.max_colwidth", 200)
(platform, ev) = next(iter(DIRS))
display(load_with_text(platform, ev).nlargest(10, "prob_hate_max")[["comment_id", "prob_hate_max", "label_source", "text"]])

,comment_id,prob_hate_max,label_source,text
389,L123036925,0.96944,herbert,Ruski morderca i g--------l Putin
3248,L123236635,0.96808,herbert,Ruscy są terrorystami nawet dla swoich hahahhahahahahah
15456,L123901107,0.96628,herbert,Obrońca Rosji kacapski troll
5173,E266069653,0.96456,herbert,"upasiona ruska świnia, zdechnie i nawet grobu mieć nie będzie, spalą go i wyrzucą pod śmietnikiem"
8809,L123557561,0.96409,herbert,ruski pet zbija kapitał polityczny na kryzysie zapoczątkowanym przez ruskie pety
17552,L124002813,0.96388,herbert,wygląda jak polski ćpun z brodą
4081,L123287677,0.96351,herbert,Cisza w c--j pisowskie klauny
15990,L123924191,0.96193,herbert,"Wyj brunatny niedorajdo,żebyś tak upadł i sobie ten głupi ryj rozwalił. Was konfedoruskie śmierdziele czuć na odległość."
8877,L123559045,0.96161,herbert,Frajerów się dyma nic nowego
1381,L123105705,0.96036,herbert,ruska onuca zdemaskowana
